# PicoNut AI Playground: LeNet-5

Gundolf Kiefer, 2026-01-21

This is an interactive environment for training, running and inspecting a [LeNet-5](https://en.wikipedia.org/wiki/LeNet), a Convolutional Neural Network (CNN) model, originally developed for recognizing hand-written digits. While frequently referenced in research articles, this implementation contains many different types of layers also found in modern, complex object recognition models: convolutional layers, fully connected layers, max pooling and ReLU activation. However, the simplicity of the network allows a complete training to be performed within seconds on a normal PC without hardware acceleration. These two aspects make the network suitable for educational purposed.

This Jupyter Notebook provides main sections to serve the following purposes:

* **Preamble:** Python code cells to be executed once after the loading the notebook. They load all necessary Python modules and initialize some data.
* **Model Definition:** PyTorch definition of the LeNet-5 model.
* **Training:** Model training.
* **Inference:** Model inference.
* **Inspection:** Cells to visualize input images and intermediate layer results.
* **C Exports:** Cells to export the trained model parameters as C files suitable for PicoNut LeNet-5 software app.
* **Scratch Area:** Room for own experiments.

All cells are editable, and you are invited to play with the parameters to learn more about the LeNet-5 model.


## Preamble

### Python Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import numpy as np

import matplotlib.pyplot as plt

### Data Loaders

Adapt `data_dir` to a suitable directory for storing downloaded data.

In [ ]:
# Directory to store downloaded data ...
data_dir = '/var/tmp/mnist-data'

# Download data and initialize loaders for training and inference ...
batch_size = 64

transform = transforms.Compose ([
    transforms.ToTensor (),
    transforms.Normalize ((0.5,), (0.5,))
])

print ("Downloading data sets ...")
train_dataset = torchvision.datasets.MNIST (root=data_dir, train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST (root=data_dir, train=False, transform=transform, download=True)

print ("Preparing loaders ...")
train_loader = torch.utils.data.DataLoader (dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader (dataset=test_dataset, batch_size=batch_size, shuffle=False)

print ("Done.")

## Model Definition

In [ ]:
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        self.conv1 = nn.Conv2d (1, 6, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d (6, 16, kernel_size=5)
        self.fc1 = nn.Linear (16*5*5, 120)
        self.fc2 = nn.Linear (120, 84)
        self.fc3 = nn.Linear (84, 10)

    def forward(self, x):
        x = torch.relu (self.conv1 (x))
        x = torch.max_pool2d (x, 2)
        x = torch.relu (self.conv2 (x))
        x = torch.max_pool2d (x, 2)
        x = x.view (-1, 16*5*5)
        x = torch.relu (self.fc1 (x))
        x = torch.relu (self.fc2 (x))
        x = self.fc3 (x)
        return x

## Training

To skip training and load a previously trained model, go to step [Load Model](#Load_Model).

### Initialization

In [ ]:
learning_rate = 0.001
num_epochs = 5

model = LeNet5 ()
criterion = nn.CrossEntropyLoss ()  # loss function
optimizer = optim.Adam (model.parameters(), lr=learning_rate)

### Training Loop

Executing this cell again without initialization will continue training.

In [ ]:
for epoch in range (num_epochs):
    model.train ()
    running_loss = 0.0
    for i, (images, labels) in enumerate (train_loader):
        optimizer.zero_grad ()
        outputs = model (images)
        loss = criterion (outputs, labels)
        loss.backward ()
        optimizer.step ()
        running_loss += loss.item ()
        if (i+1) % 100 == 0:
            print (f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {running_loss/100:.4f}')
            running_loss = 0.0

    # Test the model ...
    model.eval ()
    correct = 0
    total = 0
    with torch.no_grad ():
        for images, labels in test_loader:
            outputs = model (images)
            _, predicted = torch.max (outputs.data, 1)
            total += labels.size (0)
            correct += (predicted == labels).sum().item()

    print (f'Accuracy of the model on the test images: {100 * correct / total}%')

print ('Training complete.')

### Save Model

Save the model for later use and reproducible inference.

In [ ]:
torch.save (model, "model.pth")
#model_scripted = torch.jit.script (model) # Export to TorchScript
#model_scripted.save ('model_scripted.pt') # Save

### Load Model

Load a previously trained model (instead of training).

In [ ]:
model = torch.load ("model.pth", weights_only=False)
model.eval()
#model = torch.jit.load('model_scripted.pt')
#model.eval()

## Inference

In [ ]:
model.eval ()
correct = 0
total = 0
with torch.no_grad ():
    for images, labels in test_loader:
        outputs = model (images)
        _, predicted = torch.max (outputs.data, 1)
        total += labels.size (0)
        correct += (predicted == labels).sum().item()

print (f'Accuracy of the model on the test images: {100 * correct / total}% ({correct} out of {total}), {total-correct} error(s)')

## Inspection

### Visualization of images

In [ ]:
# Helper to show a batch of images together with labels ...
def show_batch (images, labels, title):
    img = torchvision.utils.make_grid (images)
    #img = img / 2 + 0.5  # unnormalize
    img = 0.5 - img / 2  # unnormalize
    img = img.permute (1, 2, 0)  # Change the order of dimensions from (C, H, W) to (H, W, C)
    plt.imshow (img)
    plt.title (title)
    plt.show ()
    print ('Ground truth:\n', ' '.join ("{}{}".format (labels[j].item(), "\n" if (j+1) % 8 == 0 else "") for j in range(batch_size)))

# Show first test image(s) ...
dataiter = iter (test_loader)
images, labels = dataiter.__next__()
show_batch (images, labels, 'Test Image(s)')

# Show all test images ...
#for images, labels in test_loader:
#    show_batch (images, labels, 'Test Images')

# Show first train images ...
#dataiter = iter (train_loader)
#images, labels = dataiter.__next__()
#show_batch (images, labels, 'Train Images')

### Visualization of Re-Imported Intermediate Features

The following cell allows to read back intermediate results generated by the PicoNut C app (`sw/apps/ai/lenet5`). Please refer to comments in the app on how to generate the required `features-<n>.dump` file(s).

In [ ]:
# Select image ...
image_idx = 0

# Get input image ...
dataiter = iter (test_loader)
images, labels = dataiter.__next__()
image = images[image_idx]
with open (f"features-{image_idx:02}.dump", "rb") as file:
    data_c = np.fromfile (file, dtype=np.float32).astype (np.float32)


# Helper to display an array of channels as tiles ...
def show_channels (tile, title):
    min = float (tile.min ())
    max = float (tile.max ())
    img = torchvision.utils.make_grid (torch.unsqueeze (tile, 1))
    img.requires_grad_(False)
    img = (max - img) / (max - min)
    #img = (max - img) / (max - min)
    #img = 0.5 - img / 2  # unnormalize
    #img = img / 2 + 0.5  # unnormalize
    img = img.permute (1, 2, 0)  # Change the order of dimensions from (C, H, W) to (H, W, C)
    plt.imshow (img)
    plt.title (f"{title} ({min:.2f} .. {max:.2f})")
    plt.show ()


# Show input image ...
show_channels (image, "Input Image")
print (image.shape)
image_c = torch.tensor (data_c[:1*32*32]).view (1, 32, 32)
data_c = data_c[1*32*32:]
show_channels (image_c, "Input Image (C)")
print (image_c.shape)

# x = torch.relu (self.conv1 (x))
conv1 = model.conv1 (image)
show_channels (conv1, "Conv1: Output")
print (conv1.shape)
conv1_relu = torch.relu (conv1)
show_channels (conv1_relu, "Conv1: ReLU")
conv1_relu_c = torch.tensor (data_c[:6*28*28]).view (6, 28, 28)
data_c = data_c[6*28*28:]
show_channels (conv1_relu_c, "Conv1: ReLU (C)")

# x = torch.max_pool2d (x, 2)
conv1_pool = torch.max_pool2d (conv1_relu, 2)
show_channels (conv1_pool, "Conv1: Max-Pool")
conv1_pool_c = torch.tensor (data_c[:6*14*14]).view (6, 14, 14)
data_c = data_c[6*14*14:]
show_channels (conv1_pool_c, "Conv1: Max-Pool (C)")
print (conv1_pool.shape)

# x = torch.relu (self.conv2 (x))
conv2 = model.conv2 (conv1_pool)
show_channels (conv2, "Conv2: Output")
print (conv2.shape)
conv2_relu = torch.relu (conv2)
show_channels (conv2_relu, "Conv2: ReLU")
conv2_relu_c = torch.tensor (data_c[:16*10*10]).view (16, 10, 10)
data_c = data_c[16*10*10:]
show_channels (conv2_relu_c, "Conv2: ReLU (C)")

# x = torch.max_pool2d (x, 2)
conv2_pool = torch.max_pool2d (conv2_relu, 2)
show_channels (conv2_pool, "Conv2: Max-Pool")
conv2_pool_c = torch.tensor (data_c[:16*5*5]).view (16, 5, 5)
data_c = data_c[16*5*5:]
show_channels (conv2_pool_c, "Conv2: Max-Pool (C)")
print (conv2_pool.shape)

# x = x.view (-1, 16*5*5)
conv2_flat = conv2_pool.view (-1, 16*5*5)
show_channels (conv2_flat.view (1, -1, 25), "Conf2: Max-Pool (flat)")
conv2_flat_c = conv2_pool_c.view (-1, 16*5*5)
show_channels (conv2_flat_c.view (1, -1, 25), "Conv2: Max-Pool (flat) (C)")
print (conv2_flat.shape)

# x = torch.relu (self.fc1 (x))
fc1 = model.fc1 (conv2_flat)
show_channels (fc1.view (1, -1, 20), "FC1: Output")
print (fc1.shape)
fc1_relu = torch.relu (fc1)
show_channels (fc1_relu.view (1, -1, 20), "FC1: ReLU")
fc1_relu_c = torch.tensor (data_c[:120])
data_c = data_c[120:]
show_channels (fc1_relu_c.view (1, -1, 20), "FC1: ReLU (C)")

# x = torch.relu (self.fc2 (x))
fc2 = model.fc2 (fc1_relu)
show_channels (fc2.view (1, -1, 14), "FC2: Output")
print (fc2.shape)
fc2_relu = torch.relu (fc2)
show_channels (fc2_relu.view (1, -1, 14), "FC2: ReLU")
fc2_relu_c = torch.tensor (data_c[:84])
data_c = data_c[84:]
show_channels (fc2_relu_c.view (1, -1, 14), "FC2: ReLU (C)")

# x = self.fc3 (x)
fc3 = model.fc3 (fc2_relu)
show_channels (fc3.view (1, -1, 10), "FC3: Output (= Final Output)")
fc3_relu_c = torch.tensor (data_c[:10])
data_c = data_c[10:]
show_channels (fc3_relu_c.view (1, -1, 10), "FC3: Output (= Final Output) (C)")


Scratch area to analyse properties of the model or intermediate results:

In [ ]:
#conv1.max()

## C Exports

### Export Images as C Array

The generated file can be used to replace the file `test_images.c` in `sw/apps/ai/lenet5`.

In [ ]:
def images_to_c (images, labels, filebase):
    print (f"  {filebase}.[h|c] ...")
    # Write header file ...
    with open (filebase + ".h", 'w') as file:
        file.write ("#pragma once\n\n" +
                    "#include <stdint.h>\n\n\n" +
                    f"#define TEST_IMAGES {len(images)}\n\n\n" +
                    "typedef struct {\n" +
                    "  int label;             /* ground truth */\n" + 
                    "  uint8_t image[32*32];  /* pixel data */\n" +
                    "} image_t;\n\n\n" + 
                    "extern image_t test_images[TEST_IMAGES];\n")

    # Write C file ...
    with open (filebase + ".c", 'w') as file:
        file.write (f"#include \"{filebase}.h\"\n\n\n" +
                    "image_t test_images[TEST_IMAGES] = { {\n")
        for n in range (len (images)):
            image2d = images[n][0]
            image2d = ((image2d + 1.0) * 127.5).int()
            label = labels[n]
            file.write (f"\n  /* Test image #{n} */\n  .label = {label}, .image = {{")
            for i in range (32):
                file.write("\n     ")
                for j in range (32):
                    if i >= 2 and i < 30 and j >= 2 and j < 30: pixel = image2d[i-2, j-2].item()
                    else: pixel = 0
                    file.write (f"{pixel:4d}")
                    if j < 31 or i < 31: file.write (",")
            if n < len (images) - 1: file.write ("\n  } }, {\n")
        file.write ("\n  } }\n};\n")

# Alternativ A: Load first batch only ...
#dataiter = iter (test_loader)
#images, labels = dataiter.__next__()

# Alternative B: Load everything ...
images = []
labels = []
for batch_of_images, batch_of_labels in test_loader:
    images.append (batch_of_images)
    labels.append (batch_of_labels)
images = torch.cat (images, dim=0)
labels = torch.cat (labels, dim=0)

# Write C files ...
print ("Exporting images as C arrays ...")
images_to_c (images, labels, "test_images")
print ("Done.")

### Export Weights and Biases to C

The generated files can be used to replace the files `model_<format>.c` in `sw/apps/ai/lenet5`.

In [ ]:
# Helper to export a vector / 1D array ...
def array_1d_to_c (file, tensor, name, dtype, factor, fmt, endl):
    width =  len (tensor)
    file.write (f"  .{name} = {{")
    for w in range (width):
        if w % 8 == 0: file.write ("\n    ")
        file.write (fmt.format (tensor[w] * factor))
        if w < width - 1: file.write (",")
    file.write ("\n  }" + endl)
    return f"  {dtype} {name}[{width}];\n"

# Helper to export a 2D array ...
def array_2d_to_c (file, tensor, name, dtype, factor, fmt, endl):
    height =  len (tensor)
    width =  len (tensor[0])
    file.write (f"  .{name} = {{")
    for w in range (width):
        file.write ("\n    {")
        for h in range (height):
            if height > 8 and h % 8 == 0: file.write ("\n      ")
            file.write (fmt.format (tensor[h][w] * factor))
            if h < height - 1: file.write (",")
        if height > 8: file.write ("\n   ")
        file.write (" }")
        if w < width - 1: file.write (",")
    file.write ("\n  }" + endl)
    return f"  {dtype} {name}[{width}][{height}];\n"

# Helper to export a 4D array ...
def array_4d_to_c (file, tensor, name, dtype, factor, fmt, endl):
    outs = len (tensor)
    channels = len (tensor[0])
    height =  len (tensor[0][0])
    width =  len (tensor[0][0][0])
    file.write (f"  .{name} = {{")
    for c in range (channels):
        file.write ("\n    {")
        for h in range (height):
            file.write ("\n      { ")
            for w in range (width):
                file.write ("\n        {")
                for o in range (outs):
                    file.write (fmt.format (tensor[o][c][h][w] * factor))
                    if o < outs - 1: file.write (",")
                file.write (" }")
                if w < width - 1: file.write (",")
            file.write ("\n      }")
            if h < height - 1: file.write (",")
        file.write ("\n    }")
        if c < channels - 1: file.write (",")
    file.write ("\n  }" + endl)
    return f"  {dtype} {name}[{channels}][{height}][{width}][{outs}];\n"

# Helper to export a complete LeNet-5 model with certain data type ...
def model_to_c (model, dtype_name, dtype, factor, fmt):
    filebase = f"model_{dtype_name}"
    print (f"  {filebase}.[h|c] ...")
    sd = model.state_dict ()
    with open (filebase + ".c", 'w') as file:
        file.write (f"#include \"{filebase}.h\"\n\n\n")
        file.write (f"const model_{dtype_name}_t model_{dtype_name} = {{\n")
        header  = array_4d_to_c (file, sd ['conv1.weight'], "conv1_weight", dtype, factor, fmt, ",\n\n")
        header += array_1d_to_c (file, sd ['conv1.bias'], "conv1_bias", dtype, factor, fmt, ",\n\n")
        header += array_4d_to_c (file, sd ['conv2.weight'], "conv2_weight", dtype, factor, fmt, ",\n\n")
        header += array_1d_to_c (file, sd ['conv2.bias'], "conv2_bias", dtype, factor, fmt, ",\n\n")
        header += array_2d_to_c (file, sd ['fc1.weight'], "fc1_weight", dtype, factor, fmt, ",\n\n")
        header += array_1d_to_c (file, sd ['fc1.bias'], "fc1_bias", dtype, factor, fmt, ",\n\n")
        header += array_2d_to_c (file, sd ['fc2.weight'], "fc2_weight", dtype, factor, fmt, ",\n\n")
        header += array_1d_to_c (file, sd ['fc2.bias'], "fc2_bias", dtype, factor, fmt, ",\n\n")
        header += array_2d_to_c (file, sd ['fc3.weight'], "fc3_weight", dtype, factor, fmt, ",\n\n")
        header += array_1d_to_c (file, sd ['fc3.bias'], "fc3_bias", dtype, factor, fmt, "\n};\n")
    with open (filebase + ".h", 'w') as file:
        file.write ("#pragma once\n\n")
        file.write ("#include <stdint.h>\n\n\n")
        file.write ("typedef struct {\n")
        file.write (header)
        file.write (f"}} model_{dtype_name}_t;\n\n\n")
        file.write (f"extern const model_{dtype_name}_t model_{dtype_name};\n");

# Export models in verious data types ...
print ("Exporting weights and biases as C arrays ...")
model_to_c (model, "float", "float", 1, " {:12.9f}")
model_to_c (model, "int16f8", "int16_t", 256, " {:6.0f}")
model_to_c (model, "int8f8", "int8_t", 256, " {:4.0f}")
#model_to_c (model, "int8f5", "int8_t", 32, " {:4.0f}")
print ("Done.")

## Scratch Area

This is a place for unsorted code snippets (provided as-is).

In [ ]:
dataiter = iter (test_loader)
images, labels = dataiter.__next__()
outputs = model (images)
torch.max (outputs, 1).indices

In [ ]:
labels

In [ ]:
model

In [ ]:
sd = model.state_dict ()
sd.keys ()

In [ ]:
sd ['conv1.weight'].shape, sd ['conv1.bias'].shape
#sd ['conv1.weight'].min (), sd ['conv1.weight'].max (), sd ['conv1.bias'].min (), sd ['conv1.bias'].max ()

In [ ]:
sd ['conv2.weight'].min (), sd ['conv2.weight'].max (), sd ['conv2.bias'].min (), sd ['conv2.bias'].max ()

In [ ]:
sd ['fc1.weight'].min (), sd ['fc1.weight'].max (), sd ['fc1.bias'].min (), sd ['fc1.bias'].max ()

In [ ]:
sd ['fc2.weight'].min (), sd ['fc2.weight'].max (), sd ['fc2.bias'].min (), sd ['fc2.bias'].max ()

In [ ]:
sd ['fc3.weight'].min (), sd ['fc3.weight'].max (), sd ['fc3.bias'].min (), sd ['fc3.bias'].max ()

In [ ]:
outs = model (images)

In [ ]:
images.shape

In [ ]:
image = images[0]
image.shape

In [ ]:
model.conv1(image).shape